# claudeknows: where is it, and how far away?

Finds water bottles, keys and shoes in photos (several per photo), draws a box
around each, and says how far away each one is (nearest first, plus rough metres).

Steps: download internet photos with boxes -> add your Drive footage -> let OWLv2
draw boxes where there are none -> train the detector -> report -> check distances
against your tape-measured photos -> try your own photo.

**Before you start:** Runtime -> Change runtime type -> choose a GPU (e.g. T4).
Then Runtime -> Run all. Expect roughly 60-90 minutes (bigger photos, 15 rounds).

Honest note: near/far order is reliable; metres are approximate (keys are the roughest).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
REPO = "https://github.com/alexg-sketch/claudeknows"
BRANCH = "main"  # change to a claude/... branch to try unmerged work
import os
if not os.path.isdir("claudeknows_repo"):
    !git clone -b $BRANCH $REPO claudeknows_repo
%cd claudeknows_repo
!git fetch -q origin $BRANCH && git reset -q --hard origin/$BRANCH  # always the latest code, even in a reused session
!git log -1 --format='Code version: %h %s'
!pip install -q -r requirements.txt transformers
!ffmpeg -version | head -1

## 2. Internet photos with boxes

Whole photos with every bottle / shoe box (Open Images, CC BY; credits are written
alongside). Key photos have no boxes there; step 4 boxes them.

In [ ]:
!python -m claudeknows.detection_data download --out data/detection --cap 300

## 3. Your own footage (optional)

From Google Drive `claudeknows_photos/`: the three object folders, `together/` (2-3 objects
in one shot) and `distance/` (photos named like `keys_1.0m.jpg`, used ONLY to test
distance, never to train). Missing folders are fine. Nothing is uploaded to GitHub.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/claudeknows_photos"
USE_DRIVE = True  # set to False only to train on internet photos and NOT save to Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Could not connect Google Drive:", e)
if USE_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("Google Drive is NOT connected, so your photos can't be used and the model "
                     "won't be saved. Run this cell again, click 'Connect to Google Drive' and tick "
                     "every box. (Or set USE_DRIVE = False to go on without Drive.)")
if os.path.isdir(DRIVE_DIR):
    print("Found", DRIVE_DIR, "with:", sorted(os.listdir(DRIVE_DIR)))
    !python -m claudeknows.detection_data import "$DRIVE_DIR" --root data/detection
else:
    print("No claudeknows_photos folder in My Drive: using internet photos only.")

## 4. Draw the missing boxes (OWLv2)

Keys and your footage get boxes from OWLv2, keeping only confident ones. The contact
sheets let you glance over them; photos where nothing confident was found are set aside.
If boxing fails, this step stops with a message instead of training on half the data.

In [ ]:
!python -m claudeknows.autobox run data/detection --threshold 0.3
# also add boxes for objects a photo has no box for (e.g. shoes behind a bottle)
!python -m claudeknows.autobox fill data/detection --threshold 0.4
import glob
from IPython.display import Image as Show, display
for folder in sorted(glob.glob("data/detection/*/keys")) + sorted(glob.glob("data/detection/own/*")):
    out = "sheet_" + folder.replace("/", "_") + ".jpg"
    !python -m claudeknows.autobox sheet "$folder" --out "$out" --n 24
    if os.path.exists(out):
        print(folder); display(Show(out))
!python -m claudeknows.detection_data summary --root data/detection
from claudeknows.detection_data import scan_detection, readiness_problems
problems = readiness_problems(*scan_detection("data/detection"))
if problems:  # stop here instead of training on half the data
    raise SystemExit("Not ready to train:\n- " + "\n- ".join(problems))
print("Ready to train: every object has boxed photos.")

## 5. Train the detector

Test photos are chosen by video / photo (never by frame). Training flips, tilts (a few degrees),
turns by exactly 90 degrees, zooms and shifts the photos with their boxes, so it learns any angle and position.

In [ ]:
EPOCHS = 15  # was 10; run 3 was still improving at the end
!python -m claudeknows.detector train --root data/detection --epochs $EPOCHS --out detector.pt

## 6. How good is it?

Accuracy (mAP@0.5) per object, internet vs your footage, far vs near, centre vs edge,
and per held-out video of yours.

In [ ]:
!python -m claudeknows.detector report --root data/detection --checkpoint detector.pt

## 6b. Experiment: mixing objects into photos (optional)

Most internet photos show only one kind of object, so this trains a second copy of the
detector that also pastes bottles / keys / shoes from other training photos into 30% of the
photos. Compare the "together/" rows below with step 6: keep this version only if they are better.

In [ ]:
!python -m claudeknows.detector train --root data/detection --epochs $EPOCHS --composite 0.3 --out detector_composite.pt
!python -m claudeknows.detector report --root data/detection --checkpoint detector_composite.pt

## 7. How far away? Check against your measured photos

Two estimates per object: a depth model (Depth Anything V2 Small) and the object's known
real size. The report says how far off they are, in plain words. Needs `distance/` photos.

In [ ]:
!python -m claudeknows.distance check --root data/detection --checkpoint detector.pt

## 8. Save the model

Saves `detector.pt` to your Google Drive in `claudeknows_models/` (one copy dated today, and
one called `detector.pt` that the quick **try_spatial** notebook loads), and also downloads it.
This runs before the try-it step, so the model is safe even if a photo upload goes wrong.

In [ ]:
import datetime, shutil
MODELS_DIR = "/content/drive/MyDrive/claudeknows_models"
if not os.path.isdir("/content/drive/MyDrive"):
    try:  # one more try, so a long training run is not lost
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Could not connect Google Drive:", e)
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(MODELS_DIR, exist_ok=True)
    dated = f"{MODELS_DIR}/detector_{datetime.date.today()}.pt"
    shutil.copy("detector.pt", dated)
    shutil.copy("detector.pt", f"{MODELS_DIR}/detector.pt")
    print("Saved to Google Drive:", dated, "and", f"{MODELS_DIR}/detector.pt")
else:
    print("Google Drive NOT connected: only downloading. Keep the downloaded detector.pt safe.")
from google.colab import files
files.download("detector.pt")

## 9. Try it: upload a photo

In [ ]:
from google.colab import files
from claudeknows.detector import load_detector, detect
from claudeknows.distance import (depth_models, open_photo, estimate_distances, describe,
                                  draw_results)
model, classes = load_detector("detector.pt")
relative_fn, metric_fn = depth_models()
up = files.upload()
for name in up:
    try:
        image, focal = open_photo(name)  # JPG, PNG or iPhone HEIC; focal is None without camera info
    except ValueError as e:
        print(e); continue
    results = estimate_distances(image, detect(model, image, classes), relative_fn, metric_fn, focal)
    print(name); print("\n".join(describe(results)) or "Nothing found.")
    display(draw_results(image, results))